<a href="https://colab.research.google.com/github/kimjiwoo2/Pill-agent/blob/develop/05_jw_20k_db_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **0. Overview**

### **Pillot DB(`drug_master`) 식별력·한계 전수 검증**

**목적** — v2까지의 "소수 클래스(빨강) 살리기" 전략을 멈추고, 속성 분류기(색·모양)가 실제 DB(drug_master, 클린 4,461종)에서 약을 어디까지 식별·압축할 수 있는지 전수 검증한다. 결과는 v3 평가 지표 전환(F1 → exp_recall@K)과 색별 투자 우선순위(3축 분석)의 근거가 된다.

**검증 항목**
1. **기초 무결성** (§0-1) — item_seq PK 유니크, 색/모양 결측 61종 제외 → 클린 4,461
2. **색상 정규화** (§1) — 원본 color_class1 → 10색 정규화 흡수 관계
3. **VALID_COMBOS** (§2) — (색,모양) 실존 조합 40개, 평균 111.5종/조합, max 하양·원형 931
4. **색별 비각인** (§3) — 27종(0.6%)만 색·모양이 유일 식별 수단 → 색 분류 임팩트 우선순위
5. **각인 커버리지** (§4) — 각인 보유 4,434(99.4%) / 비각인 27 / 그중 무글자각인(마크·분할선만) 84
6. **color2 ROI** (§5) — 보유 19%뿐, 거대 조합엔 정보 부재 → color2 헤드 제외
7. **line_front** (§6) — 89% 빈값·품질 의심 → line 헤드 제외
8. **헤드별 후보압축** (§7) — color만 446종 → +shape 111종(현 헤드) → +color2 39.5종(착시)
9. **충돌 종** (§9) — 색·모양·양면 글자각인 다 같아 OCR로도 못 가르는 이론적 하한
10. **외부 DB 키** (§10) — DUR·e약은요 매핑 키(di_edi_code 등) 점검

**핵심 결론** — 속성 분류기는 식별기가 아니라 **후보 압축기**(평균 111종까지). 최종 식별은 OCR 몫이나, OCR도 무글자각인(84)·비각인(27) = 111종(2.5%)엔 무력 → 이것이 파이프라인의 절대 한계. 데모는 이 한계 밖(잘 맞는 색 ∩ 글자 각인)으로 스코핑한다.

## **0-1. 연결 · 스키마 · 기초 무결성**

SQLAlchemy 엔진으로 연결 . PK 무결성도 함께 확인.

In [ ]:
!pip -q install pymysql sqlalchemy

import pandas as pd
from sqlalchemy import create_engine

USER = "jiwoo_admin"
PW   = "1234"
HOST = "103.218.161.72"
DB   = "pilliot_db"
engine = create_engine(f"mysql+pymysql://{USER}:{PW}@{HOST}/{DB}?charset=utf8mb4")

def q(sql):
    return pd.read_sql(sql, engine)

pd.set_option("display.max_rows", 120)

# 스키마
schema = q("DESCRIBE drug_master")
print(schema[["Field", "Type"]].to_string(index=False))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 532.4 kB/s eta 0:00:00
          Field         Type
       item_seq       bigint
dl_mapping_code varchar(100)
        dl_name varchar(500)
     dl_name_en varchar(500)
     dl_company varchar(500)
    dl_material         text
 dl_material_en         text
     drug_shape varchar(100)
   color_class1  varchar(50)
   color_class2  varchar(50)
 form_code_name varchar(100)
    print_front varchar(100)
     print_back varchar(100)
     line_front varchar(100)
      line_back varchar(100)
          chart         text
    di_class_no         text
di_etc_otc_code varchar(100)
    di_edi_code         text
     created_at    timestamp
     updated_at    timestamp


In [ ]:
# 기초 무결성 — item_seq PK 유니크 / 중복
print(q("""
    SELECT COUNT(*) AS total,
           COUNT(DISTINCT item_seq) AS uniq_item_seq,
           COUNT(*) - COUNT(DISTINCT item_seq) AS dup_count
    FROM drug_master
""").to_string(index=False))

# 중복 있으면 표시 (없으면 빈 결과)
dup = q("""
    SELECT item_seq, COUNT(*) c FROM drug_master
    GROUP BY item_seq HAVING c > 1 ORDER BY c DESC LIMIT 20
""")
print("\n중복 item_seq:", "없음" if dup.empty else "")
if not dup.empty:
    print(dup.to_string(index=False))

 total  uniq_item_seq  dup_count
  4522           4522          0

중복 item_seq: 없음


In [ ]:
# 원본 raw 값 전수 — 정규화 매핑 근거 (color1/color2/shape/print_back 포함)
for col in ["color_class1", "color_class2", "drug_shape", "print_back"]:
    print(f"\n[{col} raw 값]")
    print(q(f"""
        SELECT {col} AS val, COUNT(*) c FROM drug_master
        GROUP BY {col} ORDER BY c DESC LIMIT 40
    """).to_string(index=False))


[color_class1 raw 값]
   val    c
    하양 1661
    노랑  577
    분홍  481
    주황  340
    갈색  323
    연두  241
    파랑  231
    초록  216
    빨강  105
  None   61
    보라   40
노랑, 투명   38
    회색   32
파랑, 투명   25
초록, 투명   24
    검정   17
빨강, 투명   16
하양, 노랑   16
    청록   15
청록, 투명   10
주황, 투명    9
    남색    8
갈색, 투명    8
    자주    8
    투명    7
보라, 투명    5
하양, 갈색    2
하양, 투명    2
분홍, 투명    2
하양, 빨강    1
하양, 파랑    1

[color_class2 raw 값]
   val    c
       3642
    하양  315
    노랑  216
  None   61
    파랑   55
    주황   35
    초록   34
    갈색   31
    연두   23
    빨강   23
    분홍   22
    투명   15
    회색   12
하양, 투명    6
파랑, 옅은    6
    보라    5
    자주    4
노랑, 옅은    3
    검정    2
갈색, 진한    2
    청록    1
연두, 진한    1
분홍, 투명    1
초록, 옅은    1
    남색    1
청록, 옅은    1
노랑, 진한    1
초록, 진한    1
갈색, 투명    1
주황, 투명    1

[drug_shape raw 값]
 val    c
  원형 1733
 장방형 1502
 타원형  962
  기타   92
 팔각형   89
None   61
 삼각형   25
 육각형   25
 오각형   17
 사각형   11
마름모형    4
 반원형    1

[print_back raw 값]
   val    c
       2307
   분할선

## **0-2. 정규화 매핑 → 분석용 마스터 `dm`**

학습 manifest 규칙과 동일:
- **color (10cls)**: 하양·노랑·분홍·갈색·파랑·주황·초록·연두·빨강 + **기타**(보라·회색·청록·검정·자주·남색·투명 등)
  - 복합색 `"갈색, 투명"` → 앞 색상만(`갈색`)
- **shape (4cls)**: 원형·타원형·장방형 + **기타**(팔각·삼각·육각·오각·사각·마름모·반원 등)
- 색/모양 결측(None·빈값)은 **학습 제외** 대상 → `None`으로 두고 분석 시 dropna


In [ ]:
# 분석에 필요한 컬럼만 로드
dm = q("""
    SELECT item_seq, color_class1, color_class2, drug_shape,
           print_front, print_back, line_front, line_back,
           di_edi_code, dl_material, form_code_name
    FROM drug_master
""")
print("로드:", dm.shape)

# ── color 정규화 (10cls) ──
COLOR_10 = {"하양","노랑","분홍","갈색","파랑","주황","초록","연두","빨강"}
def norm_color(v):
    if pd.isna(v) or str(v).strip() == "":
        return None
    first = str(v).split(",")[0].strip()      # "갈색, 투명" -> "갈색"
    return first if first in COLOR_10 else "기타"
dm["color_norm"] = dm["color_class1"].apply(norm_color)

# ── color2 정규화 (보조색, 분석용) ──
def norm_color2(v):
    if pd.isna(v) or str(v).strip() == "":
        return "none"                          # 보조색 없음
    first = str(v).split(",")[0].strip()
    return first if first in COLOR_10 else "기타"
dm["color2_norm"] = dm["color_class2"].apply(norm_color2)

# ── shape 정규화 (4cls) ──
SHAPE_4 = {"원형","타원형","장방형"}
def norm_shape(v):
    if pd.isna(v) or str(v).strip() == "":
        return None
    s = str(v).strip()
    return s if s in SHAPE_4 else "기타"
dm["shape_norm"] = dm["drug_shape"].apply(norm_shape)

# ── 비각인 플래그 (앞/뒤 각인 둘 다 비었으면 비각인) ──
def is_blank(s): return pd.isna(s) or str(s).strip() == ""
dm["no_engraving"] = dm.apply(
    lambda r: is_blank(r["print_front"]) and is_blank(r["print_back"]), axis=1)

# 검증
print("\ncolor_norm 분포:\n", dm["color_norm"].value_counts(dropna=False))
print("\nshape_norm 분포:\n", dm["shape_norm"].value_counts(dropna=False))
print("\n색 결측(학습 제외):", dm["color_norm"].isna().sum())
print("모양 결측:", dm["shape_norm"].isna().sum())
print("비각인 종 수:", int(dm["no_engraving"].sum()))

# 분석용 클린본 (색/모양 결측 제외) — 이후 조합 분석은 이걸로
dmc = dm.dropna(subset=["color_norm","shape_norm"]).copy()
print("\n분석 클린본 dmc:", dmc.shape)

로드: (4522, 11)

color_norm 분포:
 color_norm
하양      1683
노랑       615
분홍       483
주황       349
갈색       331
파랑       256
연두       241
초록       240
기타       142
빨강       121
None      61
Name: count, dtype: int64

shape_norm 분포:
 shape_norm
원형      1733
장방형     1502
타원형      962
기타       264
None      61
Name: count, dtype: int64

색 결측(학습 제외): 61
모양 결측: 61
비각인 종 수: 88

분석 클린본 dmc: (4461, 15)


## **1. 색상 정규화 전후 종 수 변화**

원본 색상값이 정규화 10색으로 어떻게 흡수됐는지 — 특히 '기타'에 무엇이 묶였는지.

In [ ]:
# 원본 color_class1 -> color_norm 흡수 관계
before_after = (dm.groupby(["color_class1","color_norm"])["item_seq"]
                  .nunique().reset_index(name="n_items")
                  .sort_values(["color_norm","n_items"], ascending=[True,False]))
print(before_after.to_string(index=False))

print("\n── '기타'로 흡수된 원본 색상 ──")
print(before_after[before_after.color_norm=="기타"].to_string(index=False))

print("\n정규화 전 원본 색상 종류 수:", dm["color_class1"].nunique())
print("정규화 후 색상 종류 수:", dmc["color_norm"].nunique())

color_class1 color_norm  n_items
          갈색         갈색      323
      갈색, 투명         갈색        8
          보라         기타       40
          회색         기타       32
          검정         기타       17
          청록         기타       15
      청록, 투명         기타       10
          남색         기타        8
          자주         기타        8
          투명         기타        7
      보라, 투명         기타        5
          노랑         노랑      577
      노랑, 투명         노랑       38
          분홍         분홍      481
      분홍, 투명         분홍        2
          빨강         빨강      105
      빨강, 투명         빨강       16
          연두         연두      241
          주황         주황      340
      주황, 투명         주황        9
          초록         초록      216
      초록, 투명         초록       24
          파랑         파랑      231
      파랑, 투명         파랑       25
          하양         하양     1661
      하양, 노랑         하양       16
      하양, 갈색         하양        2
      하양, 투명         하양        2
      하양, 빨강         하양        1
      하양, 

## **2. (color, shape) 조합 전수 + VALID_COMBOS 확인**

정규화 후 색 10 × 모양 4 class 기준, 최대 40개 조합에서 실제 약이 존재하는 조합만 = VALID_COMBOS.

In [ ]:
combo = (dmc.groupby(["color_norm","shape_norm"])["item_seq"]
            .nunique().reset_index(name="n_items")
            .sort_values("n_items", ascending=False))

print("정규화 후 실제 존재 조합 수 (VALID_COMBOS):", len(combo))
print("이론상 최대(10x4):", 10*4)
print("avg items/combo:", round(combo.n_items.mean(),1),
      "| median:", int(combo.n_items.median()),
      "| max:", int(combo.n_items.max()))
print()
print(combo.to_string(index=False))   # 하양원형 최다 확인

정규화 후 실제 존재 조합 수 (VALID_COMBOS): 40
이론상 최대(10x4): 40
avg items/combo: 111.5 | median: 77 | max: 931

color_norm shape_norm  n_items
        하양         원형      931
        하양        장방형      384
        분홍         원형      260
        하양        타원형      253
        노랑         원형      211
        노랑        장방형      206
        갈색        장방형      179
        초록        장방형      169
        노랑        타원형      149
        주황         원형      137
        주황        장방형      124
        파랑        장방형      124
        갈색        타원형      115
        하양         기타      115
        분홍        타원형      111
        분홍        장방형       83
        연두        타원형       82
        기타        장방형       80
        빨강        장방형       79
        주황        타원형       78
        연두         원형       76
        연두        장방형       74
        파랑        타원형       71
        노랑         기타       49
        기타        타원형       36
        초록        타원형       34
        빨강        타원형       33
        갈색         원형       32


In [ ]:
# 조합 매트릭스(피벗) — 빈칸 = 약 없는 조합
pivot = (dmc.pivot_table(index="color_norm", columns="shape_norm",
                         values="item_seq", aggfunc="nunique", fill_value=0))
print(pivot.to_string())
print("\n0인 조합 수(존재하지 않는 색x모양):", int((pivot==0).sum().sum()))

shape_norm   기타   원형  장방형  타원형
color_norm                    
갈색            5   32  179  115
기타            8   18   80   36
노랑           49  211  206  149
분홍           29  260   83  111
빨강            1    8   79   33
연두            9   76   74   82
주황           10  137  124   78
초록            8   29  169   34
파랑           30   31  124   71
하양          115  931  384  253

0인 조합 수(존재하지 않는 색x모양): 0


## **3. 색별 비각인 종 수 — 색 분류 임팩트 우선순위**

비각인 = 각인 없음 → **OCR로 못 잡음 → 색·모양 분류가 유일한 식별 수단**.
비각인 종이 많은 색일수록 색 분류 임팩트가 크다.

In [ ]:
eng = (dmc.groupby("color_norm")
          .agg(total_items=("item_seq","nunique"),
               no_eng_items=("no_engraving","sum"))
          .reset_index())
eng["no_eng_items"] = eng["no_eng_items"].astype(int)
eng["no_eng_pct"] = (100*eng.no_eng_items/eng.total_items).round(1)
eng = eng.sort_values("no_eng_items", ascending=False)
print(eng.to_string(index=False))
print("\n전체 비각인 종 수:", int(dmc["no_engraving"].sum()))

color_norm  total_items  no_eng_items  no_eng_pct
        하양         1683            11         0.7
        노랑          615             8         1.3
        갈색          331             3         0.9
        기타          142             2         1.4
        주황          349             1         0.3
        초록          240             1         0.4
        파랑          256             1         0.4
        연두          241             0         0.0
        분홍          483             0         0.0
        빨강          121             0         0.0

전체 비각인 종 수: 27


> **해석 메모**: 빨강 0 / 주황 1처럼 비각인 0~1인 색은 OCR이 100% 커버 → 색분류 임팩트 ≈ 0 (디프라이오).
> 분홍·초록·연두 등 비각인 종 수가 의미 있게 큰 색이 색분류로 손볼 가치 있는 클래스.

## **4. 각인(print) 커버리지 — OCR 적용 가능 범위**

In [ ]:
total = dmc["item_seq"].nunique()
no_eng = int(dmc["no_engraving"].sum())
has_eng = total - no_eng
print(f"전체 종: {total}")
print(f"각인 보유: {has_eng} ({100*has_eng/total:.1f}%)  <- OCR 적용 가능")
print(f"비각인  : {no_eng} ({100*no_eng/total:.1f}%)  <- 색/모양으로만 식별")

전체 종: 4461
각인 보유: 4434 (99.4%)  <- OCR 적용 가능
비각인  : 27 (0.6%)  <- 색/모양으로만 식별


## **5. color2(보조색) 영향 — 조합 풀 확장 ROI**

color2 헤드를 추가하면 (color, shape) → (color, color2, shape)로 후보가 더 좁혀지는지.

In [ ]:
combo2 = (dmc.groupby(["color_norm","color2_norm","shape_norm"])["item_seq"]
             .nunique().reset_index(name="n_items"))
print("2D (color,shape) 조합 수      :", len(combo))
print("3D (color,color2,shape) 조합 수:", len(combo2))
print()
print("2D avg items/combo:", round(combo.n_items.mean(),1))
print("3D avg items/combo:", round(combo2.n_items.mean(),1))
print("\ncolor2 보유 비율:",
      f'{100*(dmc.color2_norm!="none").mean():.1f}%')


2D (color,shape) 조합 수      : 40
3D (color,color2,shape) 조합 수: 113

2D avg items/combo: 111.5
3D avg items/combo: 39.5

color2 보유 비율: 18.4%


In [ ]:
# (1) color2 보유 종이 어느 (color, shape) 조합에 분포하나
has_c2 = dmc[dmc.color2_norm != "none"].copy()
print(f"color2 보유 종: {len(has_c2)} / {len(dmc)} ({100*len(has_c2)/len(dmc):.1f}%)")
print()

dist = (has_c2.groupby(["color_norm","shape_norm"])["item_seq"]
          .nunique().reset_index(name="c2_items")
          .sort_values("c2_items", ascending=False))

# 각 조합의 전체 종수 대비 color2 보유 비율도 함께
combo_total = (dmc.groupby(["color_norm","shape_norm"])["item_seq"]
                 .nunique().reset_index(name="total_items"))
dist = dist.merge(combo_total, on=["color_norm","shape_norm"])
dist["c2_coverage_%"] = (100*dist.c2_items/dist.total_items).round(1)
print(dist.head(20).to_string(index=False))

color2 보유 종: 819 / 4461 (18.4%)

color_norm shape_norm  c2_items  total_items  c2_coverage_%
        초록        장방형       143          169           84.6
        하양        장방형       121          384           31.5
        노랑        장방형        97          206           47.1
        갈색        장방형        93          179           52.0
        파랑        장방형        93          124           75.0
        주황        장방형        68          124           54.8
        기타        장방형        68           80           85.0
        빨강        장방형        60           79           75.9
        하양         원형        20          931            2.1
        분홍        장방형        16           83           19.3
        하양        타원형        15          253            5.9
        연두        장방형        14           74           18.9
        갈색        타원형         4          115            3.5
        연두        타원형         3           82            3.7
        파랑        타원형         3           71            4.2
       

In [ ]:
# (2) color2 넣으면 거대 조합이 실제로 쪼개지나
#  하양원형(931) 같은 큰 조합 안에서 color2별 분포
for cc, ss in [("하양","원형"), ("하양","장방형"), ("분홍","원형"), ("노랑","원형")]:
    sub = dmc[(dmc.color_norm==cc) & (dmc.shape_norm==ss)]
    vc = sub["color2_norm"].value_counts()
    none_n = vc.get("none", 0)
    print(f"\n[{cc}·{ss}] 총 {len(sub)}종")
    print(f"  color2 없음(none): {none_n} ({100*none_n/len(sub):.0f}%)")
    print(f"  color2 있는 것: {vc.drop('none', errors='ignore').to_dict()}")


[하양·원형] 총 931종
  color2 없음(none): 911 (98%)
  color2 있는 것: {'노랑': 14, '하양': 4, '파랑': 1, '분홍': 1}

[하양·장방형] 총 384종
  color2 없음(none): 263 (68%)
  color2 있는 것: {'하양': 117, '기타': 1, '노랑': 1, '주황': 1, '파랑': 1}

[분홍·원형] 총 260종
  color2 없음(none): 260 (100%)
  color2 있는 것: {}

[노랑·원형] 총 211종
  color2 없음(none): 211 (100%)
  color2 있는 것: {}


## **6. line_front(분할선) 분포 — line 헤드 비채택 근거 박제**

In [ ]:
ln = (dm.assign(line_blank=dm.line_front.apply(is_blank))
        .groupby(dm.line_front.fillna("(None)"))["item_seq"]
        .nunique().reset_index(name="n_items")
        .sort_values("n_items", ascending=False))
ln.columns = ["line_front","n_items"]
ln["pct"] = (100*ln.n_items/ln.n_items.sum()).round(1)
print(ln.to_string(index=False))

blank = dm["line_front"].apply(is_blank).mean()
print(f"\nline_front 빈값/None 비율: {100*blank:.1f}%  -> 정보량 부족, 헤드 비채택")

line_front  n_items  pct
               3972 87.8
         -      478 10.6
    (None)       61  1.3
         +        8  0.2
        기타        3  0.1

line_front 빈값/None 비율: 89.2%  -> 정보량 부족, 헤드 비채택


## **7. 헤드별 후보압축 시뮬레이션 ★현 헤드로 어디까지 식별 가능한가★**

각 헤드 조합으로 group했을 때 **평균 후보 종 수** = 그 헤드가 후보를 얼마나 좁히는가.
색만 → +모양 → +보조색 순으로 ROI가 한눈에.

In [ ]:
def compress(cols, label):
    g = dmc.groupby(cols)["item_seq"].nunique()
    return {"단계": label, "n_combos": len(g),
            "avg_cand": round(g.mean(),1),
            "median_cand": int(g.median()),
            "max_cand": int(g.max())}

rows = [
    compress(["color_norm"],                              "color만"),
    compress(["color_norm","shape_norm"],                 "color+shape (현재)")
]
sim = pd.DataFrame(rows)
print(sim.to_string(index=False))
print(f"\n전체 종 {dmc.item_seq.nunique()}개 기준,")
print("avg_cand = 그 헤드 조합 하나가 평균 몇 종으로 좁히는가 (작을수록 식별력 ↑)")

                단계  n_combos  avg_cand  median_cand  max_cand
            color만        10     446.1          293      1683
  color+shape (현재)        40     111.5           77       931
color+shape+color2       113      39.5            9       911

전체 종 4461개 기준,
avg_cand = 그 헤드 조합 하나가 평균 몇 종으로 좁히는가 (작을수록 식별력 ↑)


## **8. 조합별 비각인 교차 — '색을 꼭 맞혀야 하는' 위험 구간**

비각인 종이 어느 (color, shape) 조합에 몰렸나. 이 조합들은 OCR이 못 도와주므로
**색·모양 분류가 유일한 무기** → 해당 색의 분류 정확도가 결정적.

In [ ]:
risk = (dmc[dmc.no_engraving]
          .groupby(["color_norm","shape_norm"])["item_seq"]
          .nunique().reset_index(name="no_eng_items")
          .sort_values("no_eng_items", ascending=False))
print("비각인이 분포한 (color,shape) 조합:")
print(risk.to_string(index=False))
print("\n색별 비각인 합:")
print(risk.groupby("color_norm").no_eng_items.sum()
        .sort_values(ascending=False).to_string())


비각인이 분포한 (color,shape) 조합:
color_norm shape_norm  no_eng_items
        하양         원형            10
        노랑         원형             4
        노랑         기타             3
        기타         원형             2
        갈색        타원형             1
        갈색        장방형             1
        갈색         기타             1
        노랑        장방형             1
        주황         원형             1
        초록         기타             1
        파랑         원형             1
        하양         기타             1

색별 비각인 합:
color_norm
하양    11
노랑     8
갈색     3
기타     2
주황     1
초록     1
파랑     1


## **9. (color, shape, 각인) 충돌 종 — 이론적 식별 불가 케이스**

헤드+OCR을 다 써도 (색·모양·앞면각인)이 동일한데 서로 다른 약 = **현 파이프라인의 절대 한계**.
이 수치가 곧 '완벽한 모델이어도 못 가르는' 하한.

In [ ]:
# 글자 각인이 있는 약만 대상으로, 양면 글자각인이 동일한 충돌
eng_only = dmc[~dmc.no_engraving].copy()
eng_only["pf"] = eng_only["print_front"].fillna("").str.strip()
eng_only["pb"] = eng_only["print_back"].fillna("").str.strip()

# 무글자(마크/분할선/빈값)는 충돌 계산에서 제외 — OCR이 어차피 못 읽음(84종 별도 집계)
NON_TEXT = {"", "마크", "분할선", "마크 분할선", "분할선 마크"}
def has_text(s): return s not in NON_TEXT
eng_only["any_text"] = eng_only["pf"].apply(has_text) | eng_only["pb"].apply(has_text)
eng_text = eng_only[eng_only["any_text"]].copy()   # 글자 각인 보유 약만

collide = (eng_text.groupby(["color_norm","shape_norm","pf","pb"])["item_seq"]
             .nunique().reset_index(name="n_items"))
collide = collide[collide.n_items > 1].sort_values("n_items", ascending=False)
print(f"[글자각인 충돌] 그룹: {len(collide)} / 얽힌 종: {int(collide.n_items.sum())}")
print("  (마크·분할선만 있는 무글자각인 84종은 제외 — 그건 충돌이 아니라 OCR 원천 무력)")
print(collide.head(30).to_string(index=False))

# 참고: 무글자각인으로 인한 무력 종도 따로
print(f"\n[무글자각인] 84종은 §4-2에서 집계 (마크/분할선뿐 → OCR 불가)")

[글자각인 충돌] 그룹: 23 / 얽힌 종: 50
  (마크·분할선만 있는 무글자각인 84종은 제외 — 그건 충돌이 아니라 OCR 원천 무력)
color_norm shape_norm             pf     pb  n_items
        하양         원형             마크      R        4
        주황         원형             마크     R5        3
        하양         기타             마크 AM분할선5        3
        노랑         원형             LE    NVR        2
        갈색        장방형       마크 PGN75               2
        갈색        타원형            MOA               2
        노랑         원형        ARICEPT     10        2
        빨강        장방형 D-W D-WMCB MCB               2
        빨강        장방형        CKD 250               2
        분홍         원형           SZR5               2
        연두        타원형             마크     60        2
        주황         원형             마크      5        2
        초록        장방형         마크 ERD               2
        주황         원형          SZR20               2
        주황         원형          SZR10               2
        하양         기타           AML5    분할선        2
        파랑        장

In [ ]:
# 단면만 읽었을 때 충돌 — 양면 촬영 가치 정량화
eng_text2 = eng_text.copy()  # 위 패치의 글자각인 보유 약

# 앞면만 읽음 (pf 기준, pb 무시)
front = (eng_text2.groupby(["color_norm","shape_norm","pf"])["item_seq"]
           .nunique().reset_index(name="n"))
front_collide = front[front.n > 1].n.sum()

# 뒷면만 읽음
back = (eng_text2.groupby(["color_norm","shape_norm","pb"])["item_seq"]
          .nunique().reset_index(name="n"))
back_collide = back[back.n > 1].n.sum()

print(f"앞면만 읽을 때 충돌: {front_collide}종")
print(f"뒷면만 읽을 때 충돌: {back_collide}종")
print(f"양면 다 읽을 때 충돌: 50종")
print(f"→ 양면 촬영으로 충돌 {front_collide}~{back_collide} → 50으로 감소")

앞면만 읽을 때 충돌: 806종
뒷면만 읽을 때 충돌: 3295종
양면 다 읽을 때 충돌: 50종
→ 양면 촬영으로 충돌 806~3295 → 50으로 감소


## **10. 외부 DB(DUR / e약은요) 매핑 키 점검**

item_seq 외에 성분/EDI 코드 등 외부 연결 가능한 키가 있는지 — RAG 단계 사전 점검.

In [ ]:
for col in ["di_edi_code", "dl_material", "form_code_name"]:
    nn = dm[col].notna().sum()
    uniq = dm[col].nunique()
    print(f"{col:16s} | non-null {nn}/{len(dm)} ({100*nn/len(dm):.0f}%) | nunique {uniq}")
print("\n-> EDI/성분 코드 채움률이 높으면 DUR(성분코드 기준) 매핑 키로 활용 가능")

di_edi_code      | non-null 4396/4522 (97%) | nunique 4396
dl_material      | non-null 4522/4522 (100%) | nunique 1367
form_code_name   | non-null 4461/4522 (99%) | nunique 31

-> EDI/성분 코드 채움률이 높으면 DUR(성분코드 기준) 매핑 키로 활용 가능


## **11. 요약 CSV export**

In [ ]:
OUT = "db_validation_outputs"
import os
os.makedirs(OUT, exist_ok=True)

combo.to_csv(f"{OUT}/combo_color_shape.csv", index=False)
eng.to_csv(f"{OUT}/no_engraving_by_color.csv", index=False)
sim.to_csv(f"{OUT}/head_compression_sim.csv", index=False)
risk.to_csv(f"{OUT}/risk_combos_no_engraving.csv", index=False)
collide.to_csv(f"{OUT}/collision_items.csv", index=False)
dm.to_csv(f"{OUT}/drug_master_normalized.csv", index=False)

print("저장 완료:", os.listdir(OUT))
engine.dispose()

저장 완료: ['risk_combos_no_engraving.csv', 'collision_items.csv', 'combo_color_shape.csv', 'drug_master_normalized.csv', 'head_compression_sim.csv', 'no_engraving_by_color.csv']
